# 03 — Multilingual (EN/DE/RU/ZH/TR/VI) original-fact unlearning evaluation

Uses saved outputs from Notebook 01 and Notebook 02. Published MQuAKE-T / MQuAKE-CF original facts; 2–4 hops, zero-shot RU/ZH/TR/VI. EN+DE SFT, EN-only unlearning.

Matched per-case accuracy, bootstrap CIs, original HotpotQA closed-book utility and W&B Forge visualizations. **A correct downstream multi-hop final answer is not proof of verbatim first-hop fact recovery.**


In [ ]:
%pip install -q "transformers>=4.48,<5" "peft>=0.15,<0.19" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2" "requests>=2.31" "datasets>=3,<5" "wandb>=0.31,<1"

In [ ]:
import os,gc,json,random,re,time,pathlib,unicodedata,math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel

MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
SEED=20261026
MAX_SEQ=384
ROOT=pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else './kaggle_output')
ROOT.mkdir(parents=True,exist_ok=True)
SYSTEM='Answer the factual question in English with only the short entity name or phrase. Do not explain. If unsure, say UNKNOWN.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def require_gpu():
    assert torch.cuda.is_available(),'Enable a GPU accelerator in Kaggle settings.'
    print('GPU devices:',[(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])

def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def get_tokenizer():
    t=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
    if t.pad_token is None:t.pad_token=t.eos_token
    return t

def get_backbone():
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=q,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')

def make_qa(e,tok):
    prompt=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':e['q']}],tokenize=True,add_generation_prompt=True)
    if isinstance(prompt,torch.Tensor):prompt=prompt.tolist()
    ans=tok.encode(e['a'],add_special_tokens=False)+[tok.eos_token_id]
    if len(prompt)+len(ans)>MAX_SEQ:
        raise ValueError(f'Example exceeds MAX_SEQ={MAX_SEQ}, increase limit or filter: {e["q"][:60]}')
    ids=prompt+ans
    return {'input_ids':torch.tensor([ids],dtype=torch.long),
            'labels':torch.tensor([[-100]*len(prompt)+ans],dtype=torch.long),
            'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}

def to_gpu(b):return {k:v.to('cuda:0') for k,v in b.items()}

def seq_logp(model,b):
    outs=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'])
    logits=outs.logits[:,:-1,:].float()
    labels=b['labels'][:,1:]
    mask=labels.ne(-100)
    selected=torch.log_softmax(logits,dim=-1).gather(-1,labels.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return (selected*mask).sum(dim=-1)

def generate_answer(model,tok,q,max_new_tokens=30):
    model.eval()
    prefix=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':q}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        out=model.generate(input_ids=prefix,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(out[0,prefix.shape[-1]:],skip_special_tokens=True).strip()

def normalize_answer(text):
    t=unicodedata.normalize('NFKC',str(text or '')).casefold().strip()
    t=re.sub(r'[^\w\s]', ' ',t)
    return re.sub(r'\s+',' ',t).strip()

def match_answer(response,gold,aliases=()):
    """Conservative exact/short-contained matching. Review outputs manually for publication."""
    p=normalize_answer(response)
    if not p or 'unknown' in p or 'i don t know' in p:return 0
    choices=[normalize_answer(x) for x in [gold]+list(aliases or []) if str(x).strip()]
    choices=[x for x in choices if x and len(x)>1]
    if p in choices:return 1
    if len(p)>75:return 0
    return int(any(re.search(r'(?<!\w)'+re.escape(c)+r'(?!\w)',p) is not None for c in choices if len(c)>2))

In [ ]:
# REAL-data multilingual evaluation, paired on original source case IDs.
require_gpu()
INPUT=pathlib.Path('/kaggle/input')
DATA_OVERRIDE=None
ADAPTER_OVERRIDES={}
EVAL_FORGET_PER_DEPTH=32   # 96 forget fact IDs, balanced 2/3/4-hop
EVAL_RETAIN_PER_DEPTH=16   # 48 retain IDs
EVAL_HOLDOUT_PER_DEPTH=16  # 48 untrained source-fact IDs
LANGUAGES=("en","de","ru","zh","tr","vi")
RUN_HOTPOT_UTILITY=True
HOTPOT_UTILITY_N=128       # public, human-written validation examples
INCLUDE_PRETRAINED=True
def one_file(name):
    fs=sorted(INPUT.rglob(name))
    if len(fs)!=1:
        raise RuntimeError(f"Expected one {name}: {fs}. Attach Notebook 01 exact completed output.")
    return fs[0]
def one_adapter(name,optional=False):
    if name in ADAPTER_OVERRIDES:
        return pathlib.Path(ADAPTER_OVERRIDES[name])
    fs=sorted(p.parent for p in INPUT.rglob("adapter_config.json")
              if p.parent.name==name+"_adapter")
    if len(fs)!=1:
        if optional and not fs: return None
        raise RuntimeError(f"Expected one {name}_adapter, found {fs}. Attach completed notebook 01/02 outputs.")
    return fs[0]

pack=json.loads((pathlib.Path(DATA_OVERRIDE) if DATA_OVERRIDE else one_file("mquake_t_records.json")).read_text())
assert pack["dataset"]=="MQuAKE-original-mixed" and pack["model_id"]==MODEL_ID and pack["schema_version"]==4
assert set(LANGUAGES)==set(pack["evaluation_languages"])
assert tuple(pack["train_languages"])==("en","de")
rng=random.Random(SEED+2026)
def sample_balanced(split, per_depth):
    selected=[]
    for depth in (2,3,4):
        group=[r for r in pack["records"] if r["split"]==split and r["depth"]==depth]
        if len(group)<per_depth:
            raise RuntimeError(f"Insufficient {split} depth {depth}: {len(group)} < {per_depth}.")
        selected.extend(rng.sample(group,per_depth))
    return selected

records=(sample_balanced("forget",EVAL_FORGET_PER_DEPTH)
        +sample_balanced("retain",EVAL_RETAIN_PER_DEPTH)
        +sample_balanced("holdout",EVAL_HOLDOUT_PER_DEPTH))
assert len({r["fact_id"] for r in records})==len(records)
adapters={x:one_adapter(x,optional=(x=="retain_only"))
          for x in ("full_sft","retain_only","ga","gd","npo")}
adapters={k:v for k,v in adapters.items() if v is not None}
assert {"full_sft","ga","gd","npo"}.issubset(adapters)
if INCLUDE_PRETRAINED:adapters={"pretrained":None,**adapters}
print("Evaluation IDs",len(records),"by split/depth",
      pd.DataFrame(records).groupby(["split","depth"]).size().to_dict())
print("Dataset SHA",pack["source_sha256"],"languages",LANGUAGES)
tokenizer=get_tokenizer()

# Independent retention benchmark. These are ORIGINAL HotpotQA validation questions,
# not generated facts; closed-book prompt WITHOUT Wikipedia evidence. NOT official HotpotQA F1.
hotpot_utility=[]
if RUN_HOTPOT_UTILITY:
    from datasets import load_dataset
    import hashlib
    from heapq import nsmallest
    dataset=load_dataset("hotpotqa/hotpot_qa","distractor",split="validation",streaming=True)
    pool=( (hashlib.sha256((str(SEED)+":"+str(row["id"])).encode()).hexdigest(),row)
           for row in dataset if str(row.get("question","")).strip() and str(row.get("answer","")).strip() )
    selected=nsmallest(HOTPOT_UTILITY_N,pool,key=lambda x:x[0])
    hotpot_utility=[{"id":row["id"],"q":row["question"],"a":row["answer"],
                     "type":row.get("type","")} for _,row in selected]
    if len(hotpot_utility)!=HOTPOT_UTILITY_N:
        raise RuntimeError("Not enough public HotpotQA validation examples")
    print("Loaded",len(hotpot_utility),"public HotpotQA validation examples")


In [ ]:
# W&B / CoreWeave Forge integration.
# Kaggle's Secrets HTTP service can fail independently of user credentials.
# Preserve all logs offline if the secret cannot be read during a Kaggle API run.
import os
from pathlib import Path
import wandb

WANDB_ENTITY = "crosslingual"
WANDB_PROJECT = "multilingual_deepunlearning"
WANDB_DASHBOARD = "https://forge.coreweave.com/wandb/crosslingual/multilingual_deepunlearning"
WANDB_GROUP = "mquake-t-20261026"
WANDB_BASE_URL = "https://api.forge.coreweave.com"
os.environ.setdefault("WANDB_BASE_URL", WANDB_BASE_URL)
WANDB_OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle").exists() else "./wandb_output")
WANDB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def start_tracking(stage, name, config=None):
    """Track online when Kaggle key works, otherwise keep syncable offline runs.

    Does not print the key. Offline runs are saved in /kaggle/working/wandb
    and can be synced later after authenticating to CoreWeave Forge.
    """
    token = os.environ.get("WANDB_API_KEY", "").strip()
    secret_problem = None
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("WANDB_API_KEY") or "").strip()
        except Exception as exc:
            secret_problem = type(exc).__name__
    if token:
        os.environ["WANDB_API_KEY"] = token
    mode = "online" if token else "offline"
    if mode == "offline":
        print(
            "[W&B] Kaggle WANDB_API_KEY not accessible"
            + (f" ({secret_problem})" if secret_problem else "")
            + ". Logging OFFLINE to /kaggle/working/wandb; training will continue."
        )
        print("[W&B] Enable the notebook's Kaggle Secrets permission for live graphs.")
    options = dict(
        entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
        job_type=stage, name=name, config=config or {}, mode=mode,
        dir=str(WANDB_OUTPUT_DIR),
        settings=wandb.Settings(init_timeout=60),
    )
    try:
        run = wandb.init(**options)
    except Exception as exc:
        if mode != "online":
            raise
        print(
            "[W&B] Online connection failed"
            f" ({type(exc).__name__}); using syncable offline logging instead."
        )
        try:
            wandb.finish(exit_code=1)
        except Exception:
            pass
        options["mode"] = "offline"
        run = wandb.init(**options)
    if run is None:
        raise RuntimeError("W&B returned no run in either online or offline mode.")
    print("[W&B]", "online" if options["mode"] == "online" else "offline",
          "run:", run.id, "| project:", WANDB_DASHBOARD)
    return run


In [ ]:
# All languages evaluate the SAME original source fact; none of RU/ZH/TR/VI are SFT supervised.
predictions=[]
hotpot_predictions=[]
for checkpoint,adapter in adapters.items():
    print("Checkpoint:",checkpoint,flush=True)
    run=start_tracking("evaluation","evaluate-"+checkpoint,{
        "checkpoint":checkpoint,"dataset":"MQuAKE-original-mixed",
        "source_sha256":pack["source_sha256"],"seed":SEED,
        "num_facts":len(records),"languages":LANGUAGES,
        "hotpotqa_n":len(hotpot_utility),
    })
    start_index=len(predictions)
    base=get_backbone()
    model=PeftModel.from_pretrained(base,str(adapter),is_trainable=False) if adapter is not None else base
    model.eval()
    for i,rec in enumerate(records):
        for lang in LANGUAGES:
            for probe in ("direct","multi"):
                stem="target" if probe=="direct" else "multi"
                q=rec[f"{stem}_question_{lang}"]
                gold=rec["target_answer"] if probe=="direct" else rec["multi_answer"]
                aliases=rec["target_aliases"] if probe=="direct" else rec["multi_aliases"]
                out=generate_answer(model,tokenizer,q)
                predictions.append({
                    "checkpoint":checkpoint,"case_id":rec["source_case_id"],
                    "fact_id":rec["fact_id"],"source_dataset":rec["source_dataset"],
                    "split":rec["split"],"depth":rec["depth"],"language":lang,
                    "probe":probe,"question":q,"gold":gold,
                    "aliases":json.dumps(aliases,ensure_ascii=False),
                    "generated_answer":out,"correct":match_answer(out,gold,aliases),
                    "source_sha256":pack["source_sha256"],
                })
        if (i+1)%20==0:
            print(checkpoint,"completed",i+1,"/",len(records),flush=True)
    if hotpot_utility:
        for item in hotpot_utility:
            generated=generate_answer(model,tokenizer,item["q"])
            hotpot_predictions.append({
                "checkpoint":checkpoint,"source_dataset":"HotpotQA/distractor/validation",
                "id":item["id"],"question":item["q"],"gold":item["a"],
                "prediction":generated,"correct":match_answer(generated,item["a"]),
                "question_type":item["type"],
            })
        hp=pd.DataFrame([x for x in hotpot_predictions if x["checkpoint"]==checkpoint])
        print(checkpoint,"HotpotQA closed-book exact/alias accuracy",hp["correct"].mean(),
              "n=",len(hp))
        run.log({"external/hotpotqa_closed_book_accuracy":float(hp["correct"].mean()),
                 "external/hotpotqa_n":len(hp),
                 "external/hotpotqa_predictions":wandb.Table(dataframe=hp)})
    frame=pd.DataFrame(predictions[start_index:])
    frame.to_csv(ROOT/f"eval_{checkpoint}_predictions.csv",index=False)
    slice_df=frame.groupby(["split","depth","language","probe"],as_index=False).agg(
        recovery=("correct","mean"),n=("correct","size"))
    run.log({"eval/recovery_by_depth_language":wandb.Table(dataframe=slice_df),
             "eval/sample_predictions":wandb.Table(dataframe=frame.sample(
                  n=min(500,len(frame)),random_state=SEED))})
    run.finish()
    del model,base
    cleanup()
if hotpot_predictions:
    pd.DataFrame(hotpot_predictions).to_csv(ROOT/"hotpotqa_closed_book_utility.csv",index=False)
df=pd.DataFrame(predictions)
df.to_csv(ROOT/"per_case_predictions.csv",index=False)
summary=df.groupby(["checkpoint","split","depth","language","probe"],as_index=False).agg(
    recovery=("correct","mean"),correct_n=("correct","sum"),count=("correct","size"))
summary.to_csv(ROOT/"recovery_summary.csv",index=False)
print(summary.to_string(index=False))


In [ ]:
# Paired, case-matched analysis:
# Base competence control is checked PER LANGUAGE and PER PROBE, for same source fact.
# Compare direct and downstream answers only after full SFT had competence.
indexed=df.set_index(["checkpoint","fact_id","language","probe"])["correct"].to_dict()
full={r["fact_id"]:r for r in records}
paired=[]
for cp in ("retain_only","ga","gd","npo"):
    if cp not in adapters:continue
    for rec in records:
        fid=rec["fact_id"]
        for lang in LANGUAGES:
            e_d=indexed[("full_sft",fid,lang,"direct")]
            e_m=indexed[("full_sft",fid,lang,"multi")]
            f_d=indexed[(cp,fid,lang,"direct")]
            f_m=indexed[(cp,fid,lang,"multi")]
            paired.append({
                "checkpoint":cp,"fact_id":fid,"split":rec["split"],
                "source_dataset":rec["source_dataset"],"depth":rec["depth"],
                "language":lang,"sft_direct":e_d,"sft_multi":e_m,
                "post_direct":f_d,"post_multi":f_m,
                "eligible_both_correct":int(e_d==1 and e_m==1),
                "direct_lost":int(e_d==1 and f_d==0),
                "downstream_recovered_after_direct_loss":
                    int(e_d==1 and e_m==1 and f_d==0 and f_m==1),
                "both_lost":int(e_d==1 and e_m==1 and f_d==0 and f_m==0),
            })
paired_df=pd.DataFrame(paired)
paired_df.to_csv(ROOT/"paired_fact_analysis.csv",index=False)
by_slice=paired_df.groupby(
    ["checkpoint","split","depth","language"],as_index=False
).agg(
    n=("fact_id","size"),n_eligible=("eligible_both_correct","sum"),
    pre_direct=("sft_direct","mean"),pre_multi=("sft_multi","mean"),
    post_direct=("post_direct","mean"),post_multi=("post_multi","mean"),
    n_direct_lost=("direct_lost","sum"),
    n_downstream_recovered=("downstream_recovered_after_direct_loss","sum"),
)
by_slice["recover_given_direct_loss"]=(
    by_slice.n_downstream_recovered /
    by_slice.n_direct_lost.replace(0,np.nan)
)
by_slice.to_csv(ROOT/"matched_language_depth.csv",index=False)

# Bootstrap paired *case IDs*, not independent samples of translated questions.
# Difference from English, conditional on baseline being correct on all four probes.
rng=np.random.default_rng(SEED+91)
bootstrap_rows=[]
for (cp,split,depth,lang),block in paired_df.groupby(
       ["checkpoint","split","depth","language"]):
    if lang=="en": continue
    local=block.set_index("fact_id")
    english=paired_df[(paired_df.checkpoint==cp)&(paired_df.split==split)
        &(paired_df.depth==depth)&(paired_df.language=="en")].set_index("fact_id")
    common=local.index.intersection(english.index)
    a,b=local.loc[common],english.loc[common]
    mask=(a.sft_direct.eq(1)&a.sft_multi.eq(1)&
          b.sft_direct.eq(1)&b.sft_multi.eq(1))
    if not mask.any():continue
    d=(a.loc[mask,"downstream_recovered_after_direct_loss"].to_numpy(dtype=float)
       -b.loc[mask,"downstream_recovered_after_direct_loss"].to_numpy(dtype=float))
    n=len(d)
    ix=rng.integers(0,n,size=(1000,n))
    samples=d[ix].mean(axis=1)
    bootstrap_rows.append({
        "checkpoint":cp,"split":split,"depth":depth,"language":lang,
        "paired_competent_n":n,"difference_vs_en":float(d.mean()),
        "ci_95_low":float(np.quantile(samples,.025)),
        "ci_95_high":float(np.quantile(samples,.975)),
    })
ci=pd.DataFrame(bootstrap_rows,columns=[
    "checkpoint","split","depth","language","paired_competent_n",
    "difference_vs_en","ci_95_low","ci_95_high"])
ci.to_csv(ROOT/"paired_bootstrap_ci.csv",index=False)
analysis_run=start_tracking("analysis","multilingual-reconstruction-analysis",{
    "source_sha256":pack["source_sha256"],"seed":SEED,
    "languages":LANGUAGES,"n_bootstrap":1000
})
analysis_run.log({
    "analysis/per_language_depth":wandb.Table(dataframe=by_slice),
    "analysis/paired_bootstrap_ci":wandb.Table(dataframe=ci),
    "analysis/recovery_summary":wandb.Table(dataframe=summary),
})
if len(by_slice):
    plot_data=by_slice[
        (by_slice.checkpoint=="npo")&(by_slice.split=="forget")
    ][["language","depth","recover_given_direct_loss"]].dropna()
    if len(plot_data):
        plot_data=plot_data.assign(language_depth=lambda d:
                         d.language+"_"+d.depth.astype(str))
        analysis_run.log({"analysis/npo_reconstruction_rates":
            wandb.plot.bar(wandb.Table(dataframe=plot_data),
             "language_depth","recover_given_direct_loss",
             title="NPO recovery after direct loss, by language and depth")})
analysis_run.finish()
print("Matched slice summary:")
print(by_slice[by_slice.split=="forget"].to_string(index=False))
print("Paired bootstrap CIs:")
print(ci[ci.split=="forget"].to_string(index=False))
print("IMPORTANT: multi-hop final answers differ from first-hop target.")
print("This estimates downstream answer persistence, NOT definitive recovery of erased intermediate fact.")


In [ ]:
# Limitations and provenance
print("Source SHA",pack["source_sha256"])
print("SFT training languages",pack["train_languages"])
print("Out-of-SFT evaluation languages",pack["crosslingual_holdout_languages"])
print("Counts by depth and language in",ROOT/"recovery_summary.csv")
print("External HotpotQA accuracy uses CLOSED-BOOK QA, not official HotpotQA F1.")
print("For research claims: audit question translations, temporal ambiguity and answer aliases.")
print("A full paper requires 3+ random seeds, unlearning strength tuning on DEV only,")
print("and explicit intermediate-recovery probes, not downstream answer alone.")


**No claimed publication result yet:** use 3+ training seeds, stronger alias/semantic adjudication, manual translation audit, real-data temporal ambiguity review, matched-bootstrap CIs, and a retention-utility evaluation. Repeating this with MLaKE or a separate published multilingual multi-hop benchmark would strengthen external validity.